In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder 

In [76]:
df = pd.read_csv("../../data/cars.csv")
df.sample(5)

,brand,km_driven,fuel,owner,selling_price
5430,Mahindra,10000,Diesel,First Owner,1450000
3424,Honda,7032,Petrol,First Owner,779000
2851,Hyundai,25000,Petrol,Third Owner,175000
5566,Hyundai,25000,Petrol,Second Owner,600000
7308,Maruti,70000,CNG,Second Owner,155000


In [77]:
df["brand"].value_counts()

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Kia                 4
Ambassador          4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [78]:
df["fuel"].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [79]:
df["owner"].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

## 1 - One-hot Encoding using pd.dummies

In [80]:
pd.get_dummies(df, columns=["fuel", "owner"], drop_first=True, dtype=np.int32)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,1,0,0,0,0,0,0
1,Skoda,120000,370000,1,0,0,0,1,0,0
2,Honda,140000,158000,0,0,1,0,0,0,1
3,Hyundai,127000,225000,1,0,0,0,0,0,0
4,Maruti,120000,130000,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,1,0,0,0,0
8124,Hyundai,119000,135000,1,0,0,1,0,0,0
8125,Maruti,120000,382000,1,0,0,0,0,0,0
8126,Tata,25000,290000,1,0,0,0,0,0,0


## 2 - One-hot Encoding using sklearn OneHotEncoder() class

In [81]:
X = df.drop(columns=["selling_price"])
y = df["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [82]:
X_train 


,brand,km_driven,fuel,owner
6518,Tata,2560,Petrol,First Owner
6144,Honda,80000,Petrol,Second Owner
6381,Hyundai,150000,Diesel,Fourth & Above Owner
438,Maruti,120000,Diesel,Second Owner
5939,Maruti,25000,Petrol,First Owner
...,...,...,...,...
5226,Mahindra,120000,Diesel,First Owner
5390,Maruti,80000,Diesel,Second Owner
860,Hyundai,35000,Petrol,First Owner
7603,Maruti,27000,Diesel,First Owner


In [83]:
ohe = OneHotEncoder(sparse_output=False, dtype= np.int32, drop='first')

X_train_ohe_1 = ohe.fit_transform(X_train[["fuel", "owner"]])
X_test_ohe_1 = ohe.transform(X_test[["fuel", "owner"]])

In [84]:
X_train_ohe_1

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 1, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 1, 0, 0]], shape=(6502, 7), dtype=int32)

In [85]:
X_train_transformed = np.hstack((X_train[["brand", "km_driven"]].values, X_train_ohe_1))
X_test_transformed = np.hstack((X_test[["brand", "km_driven"]].values, X_test_ohe_1))

In [86]:
X_train_transformed

array([['Tata', 2560, 0, ..., 0, 0, 0],
       ['Honda', 80000, 0, ..., 1, 0, 0],
       ['Hyundai', 150000, 1, ..., 0, 0, 0],
       ...,
       ['Hyundai', 35000, 0, ..., 0, 0, 0],
       ['Maruti', 27000, 1, ..., 0, 0, 0],
       ['Maruti', 70000, 0, ..., 1, 0, 0]], shape=(6502, 9), dtype=object)

## 3- One Hot Encoding when there are too many categories in a feature and we want to club those with less frequency in others feature

In [87]:
counts = X_train["brand"].value_counts()

In [88]:
threshold = 100
# Those brands with less than 100 counts will be grouped into "Other"

In [89]:
replace = counts[counts < threshold].index
replace

Index(['BMW', 'Skoda', 'Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz',
       'Fiat', 'Audi', 'Jeep', 'Lexus', 'Mitsubishi', 'Force', 'Land', 'Kia',
       'Ambassador', 'MG', 'Daewoo', 'Isuzu', 'Ashok', 'Peugeot', 'Opel'],
      dtype='str', name='brand')

In [90]:
pd.get_dummies(df['brand'].replace(replace, 'Others'), drop_first=True, dtype=np.int32)

,Ford,Honda,Hyundai,Mahindra,Maruti,Others,Renault,Tata,Toyota,Volkswagen
0,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
8123,0,0,1,0,0,0,0,0,0,0
8124,0,0,1,0,0,0,0,0,0,0
8125,0,0,0,0,1,0,0,0,0,0
8126,0,0,0,0,0,0,0,1,0,0


In [91]:
#One-hot encoding for features like brand must be done before splitting the data into train and test sets. 
#This is because the test set may contain brands that are not present in the training set, leading to issues during model training and evaluation.
# By performing one-hot encoding before the split, we ensure that all possible categories are accounted for in both the training and test sets, allowing for consistent feature representation.